# Classification des types de globules blancs

Ce notebook propose une cha?ne de traitement compl?te pour identifier automatiquement les leucocytes (lymphocytes, monocytes, neutrophiles, ?osinophiles, basophiles, etc.) ? partir d'images de microscopie optique.- **Objectif** : entra?ner un classifieur profond capable de diff?rencier des cellules pr?sentant des morphologies tr?s proches.- **Jeu de donn?es** : Blood Cell Images (~12 500 images coloris?es) d?j? r?parties par classe dans des dossiers `train/` et `test/`, compl?t?s par un jeu d'entra?nement augment?.- **D?fi principal** : forte similarit? visuelle entre certains types de globules blancs (ex. monocytes vs lymphocytes) et variabilit? des conditions d'acquisition (coloration, luminosit?, focus).- **Plan** : import des d?pendances, pr?paration des donn?es avec augmentation, conception d'un CNN sp?cifique, entra?nement/?valuation puis visualisation des pr?dictions.


## 2. Import des librairies et configuration


In [1]:

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use('seaborn-v0_8')
sns.set_context('talk')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}')
print('GPU(s) disponible(s) :', tf.config.list_physical_devices('GPU'))


ModuleNotFoundError: No module named 'numpy'

## 3. Chargement des donnees (train/test)


In [2]:

DATA_ROOT = Path('data') / 'blood_cell_images'
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        'Les dossiers train/test sont introuvables. V?rifiez le chemin data/blood_cell_images.'
    )

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15

base_train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

base_val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

base_test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = base_train_ds.class_names
num_classes = len(class_names)

print(f'Classes d?tect?es ({num_classes}) : {class_names}')
print(f'Taille des images : {IMG_SIZE} - Batch size : {BATCH_SIZE}')


FileNotFoundError: Les dossiers train/test sont introuvables. V?rifiez le chemin data/blood_cell_images.

In [3]:

train_count = len(base_train_ds.file_paths)
val_count = len(base_val_ds.file_paths)
test_count = len(base_test_ds.file_paths)

def describe_split(name, count):
    pct = count / (train_count + val_count + test_count) * 100
    print(f'{name:<10} -> {count:5d} images ({pct:4.1f}%)')

print('Repartition des images :')
describe_split('Train', train_count)
describe_split('Validation', val_count)
describe_split('Test', test_count)


NameError: name 'base_train_ds' is not defined

## 4. Pretraitement des images et augmentation de donnees


In [4]:

AUTOTUNE = tf.data.AUTOTUNE

# S?quence d'augmentation appliqu?e ? la vol?e (rotation/zoom/shift/flip/contraste)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.12),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
], name='data_augmentation')

normalization_layer = layers.Rescaling(1.0 / 255)

def prepare_dataset(dataset, augment=False):
    def _process(images, labels):
        if augment:
            images = data_augmentation(images, training=True)
        images = normalization_layer(images)
        return images, labels
    return dataset.map(_process, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

train_ds = prepare_dataset(base_train_ds, augment=True)
val_ds = prepare_dataset(base_val_ds, augment=False)
test_ds = prepare_dataset(base_test_ds, augment=False)


NameError: name 'tf' is not defined

## 5. Construction du modele CNN


In [5]:

def conv_block(x, filters, dropout_rate=0.2):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    return x

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = conv_block(inputs, 32, dropout_rate=0.1)
x = conv_block(x, 64, dropout_rate=0.15)
x = conv_block(x, 128, dropout_rate=0.2)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='wbc_cnn_classifier')
model.summary()


NameError: name 'keras' is not defined

## 6. Compilation et entrainement du modele


In [ ]:

EPOCHS = 30
LEARNING_RATE = 1e-4

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

checkpoint_dir = Path('models') / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'wbc_cnn.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
    ),
]

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
)


### Courbes d'apprentissage (accuracy & loss)


In [ ]:

acc = history.history.get('accuracy', [])
val_acc = history.history.get('val_accuracy', [])
loss = history.history.get('loss', [])
val_loss = history.history.get('val_loss', [])
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()


## 7. evaluation sur le jeu de test


In [ ]:

test_loss, test_acc = model.evaluate(test_ds)
print(f'Loss test : {test_loss:.4f}')
print(f'Accuracy test : {test_acc:.4f}')

y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

conf_mat = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('V?rit? terrain')
plt.xlabel('Pr?diction')
plt.title('Matrice de confusion')
plt.tight_layout()
plt.show()

report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print('Classification report:
')
print(report)


## 8. Visualisation des resultats (predictions vs verit)


In [ ]:

def show_sample_predictions(dataset, n_images=9):
    sample_batch = next(iter(dataset.unbatch().batch(n_images)))
    images, labels = sample_batch
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)

    images = images.numpy()
    labels = labels.numpy()
    total = min(n_images, images.shape[0])

    cols = int(np.ceil(np.sqrt(total)))
    rows = int(np.ceil(total / cols))
    plt.figure(figsize=(3 * cols, 3 * rows))
    for idx in range(total):
        ax = plt.subplot(rows, cols, idx + 1)
        plt.imshow(images[idx])
        pred_name = class_names[pred_labels[idx]]
        true_name = class_names[labels[idx]]
        color = 'green' if pred_name == true_name else 'red'
        plt.title(f'P: {pred_name}
T: {true_name}', color=color)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_predictions(test_ds, n_images=9)
